# 12 · Eğitim ve Metin Üretimi

**Belgedeki bölüm:** Embedding › değerler nasıl öğrenilir (loss, geri yayılım); DataLoader; GPT Gibi Modeller › Üretim

- Veri: `text.txt` → token'lar → bir token kaydırılmış (girdi, hedef) çiftleri.
- Loss: CrossEntropy, yani doğru token'a verilen olasılığın −log'u.
- Geri yayılım + optimizer (AdamW) ağırlıkları günceller; loss düşer.
- Eğitilen model kaydedilir, yüklenir ve metin üretir.

In [ ]:
import torch
from usta_model import UstaModel
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "the capital of the united"

tokens = u_tokenizer.encode(prompt)
tokens

In [ ]:
context_length = 32

In [ ]:
torch.manual_seed(1)
u_model = UstaModel(vocab_size=len(u_tokenizer.vocab), embedding_dim=32, num_heads=4, context_length=context_length, num_layers=2)

out = u_model(tokens)
out.shape

In [ ]:
out = u_model.generate(tokens, 3)
u_tokenizer.decode(out)

In [ ]:
with open("text.txt", "r") as f:
  text = f.read()

len(text), text[:100]

In [ ]:
token_ids = u_tokenizer.encode(text)
len(token_ids), type(token_ids)

In [ ]:
ids = token_ids.detach().cpu().numpy().tolist()
len(ids), type(ids)

In [ ]:
from text_dataset import TextDataset

stride = 12

dataset = TextDataset(ids, context_length, stride)

len(dataset.inputs), len(dataset.targets)

In [ ]:
dataset.inputs[0], dataset.targets[0]

In [ ]:
# model parameters count
parameters_count = sum(p.numel() for p in u_model.parameters())
print(parameters_count)

# model architecture
print(u_model)

In [ ]:
u_model.embedding.embedding.weight.shape

In [ ]:
out0 = u_model(dataset.inputs[0])
out0.shape

In [ ]:
import torch.nn as nn

loss_fn = nn.CrossEntropyLoss()

In [ ]:
loss = loss_fn(out0, dataset.targets[0])
loss

In [ ]:
loss.item()

In [ ]:
# optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
optimizer = torch.optim.AdamW(u_model.parameters(), lr=1e-3)

In [ ]:
for input, target in dataset:
  print(input.shape, target.shape)
  break

In [ ]:
u_model.train()  # dropout eğitimde açık

In [ ]:
epoch = 20

for epoch in range(epoch):
  total_loss = 0.
  for input, target in dataset:
    pred = u_model(input)
    
    loss = loss_fn(pred, target)
    total_loss += loss.item()
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

  average_loss = total_loss / len(dataset)
  print(f"Epoch {epoch + 1} loss: {loss.item()} average loss: {average_loss}")
    

### Eğitilmiş modelle tahmin

In [ ]:
u_model.eval()  # çıkarımda dropout kapalı

In [ ]:
import torch

new_tokens = u_tokenizer.encode("madrid is in")
new_tokens = new_tokens.detach().cpu().numpy().tolist()
new_tokens.append(61)

out = u_model(torch.tensor(new_tokens))

probs = torch.softmax(out[-1], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

In [ ]:
u_tokenizer.decode([max_index.item()])

### Kaydet, yükle, metin üret

In [ ]:
# save model
torch.save(u_model.state_dict(), "u_model.pth")

# load model
u_model.load_state_dict(torch.load("u_model.pth"))

# generate text
new_tokens = u_tokenizer.encode("the capital of the united states is london. the capital of france is")
new_tokens = new_tokens.detach().cpu().numpy().tolist()
new_tokens.append(61)
len(new_tokens)

In [ ]:
loaded_model = UstaModel(64, embedding_dim=32, num_heads=4, context_length=32, num_layers=2)
loaded_model.load_state_dict(torch.load("u_model.pth"))
loaded_model.eval()

In [ ]:
out = u_model(torch.tensor(new_tokens))

probs = torch.softmax(out[-1], dim=-1)
max_prob, max_index = torch.max(probs, dim=-1)
max_prob, max_index, probs

In [ ]:
generated = u_model.generate(torch.tensor(new_tokens), 5)
u_tokenizer.decode(generated)